# Smoothing Noisy Sensor Data

Sensors are never perfect. Even a good altimeter on a model rocket will report numbers that jiggle around the *true* altitude — that jiggle is called **noise**.

A **filter** is a small piece of math that takes a noisy signal and tries to recover a smoother estimate of what's really going on. In this notebook we'll look at three of them:

1. **SMA** — Simple Moving Average
2. **EMA** — Exponential Moving Average *(this is the classic recursive first-order filter)*
3. **Kalman filter** — a smarter filter that uses a model of how the rocket actually flies

At the bottom of the notebook there's an interactive plot where you can play with the parameters and see how each filter reacts.

---

### Our test signal

We simulate a model rocket: a short powered boost, then a ballistic coast up and back down. We add Gaussian noise to fake an imperfect altimeter, and use this as the input to our filters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Checkbox

%matplotlib inline

# Generate the true altitude curve of a model rocket: boost + coast.
g       = 9.81   # gravity (m/s^2)
a_boost = 80     # boost acceleration (m/s^2)
t_burn  = 0.8    # motor burn time (s)

t  = np.arange(0, 15, 0.02)   # 50 Hz sampling
dt = t[1] - t[0]

boost = t <= t_burn
true_sig = np.empty_like(t)
true_sig[boost] = 0.5 * (a_boost - g) * t[boost]**2

h_burn = 0.5 * (a_boost - g) * t_burn**2
v_burn = (a_boost - g) * t_burn
tc = t[~boost] - t_burn
true_sig[~boost] = h_burn + v_burn*tc - 0.5*g*tc**2
true_sig = np.maximum(true_sig, 0)

# Trim to the moment the rocket lands.
landed = np.where((true_sig == 0) & (t > t_burn))[0]
if len(landed):
    end = landed[0] + 1
    t = t[:end]; true_sig = true_sig[:end]

# A reproducible noise sequence we'll scale later.
rng = np.random.default_rng(0)
unit_noise = rng.standard_normal(t.size)

## 1. Simple Moving Average (SMA)

The simplest idea: **average the last $N$ samples**.

$$
y_n \;=\; \frac{1}{N}\bigl(x_n + x_{n-1} + x_{n-2} + \dots + x_{n-N+1}\bigr)
$$

- A bigger window $N$ → smoother output, but **slower** to react to changes.
- A small window → reacts fast but stays noisy.
- The filter has to remember the last $N$ samples, so it costs memory.

In [ ]:
def sma(x, window):
    padded = np.concatenate([np.full(window-1, x[0]), x])
    return np.convolve(padded, np.ones(window)/window, mode='valid')

## 2. Exponential Moving Average (EMA)

This is the classic **recursive first-order filter**. Instead of remembering many samples, it only remembers its *previous output*:

$$
y_n \;=\; \alpha\, x_n \;+\; (1 - \alpha)\, y_{n-1}
$$

where $0 < \alpha \le 1$ is the **smoothing factor**.

- $\alpha$ close to $1$ → trust the new measurement, react fast, less smoothing.
- $\alpha$ close to $0$ → trust the old estimate, react slowly, lots of smoothing.

Why is it called *recursive*? Because the new output $y_n$ is defined using the *previous output* $y_{n-1}$ — the formula refers to itself. That's also why it only needs to store one number, no matter how long the signal is.

In [ ]:
def ema(x, alpha):
    y = np.empty_like(x)
    y[0] = x[0]
    for i in range(1, len(x)):
        y[i] = alpha*x[i] + (1-alpha)*y[i-1]
    return y

## 3. Kalman filter

The two filters above know nothing about the rocket — they just smooth numbers. A **Kalman filter** is smarter: it uses a *model* of how the system moves.

### Our model: constant velocity

We track two things at every moment: altitude $h$ and vertical velocity $v$.

$$
\mathbf{x} = \begin{bmatrix} h \\ v \end{bmatrix}
$$

Between two samples (a time step $\Delta t$ apart) we *predict* the next state assuming velocity stays constant:

$$
h_{n+1} = h_n + v_n \, \Delta t \qquad v_{n+1} = v_n
$$

In matrix form:

$$
\mathbf{x}_{n+1} \;=\; F\, \mathbf{x}_n, \qquad F = \begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix}
$$

Of course velocity is *not* constant (gravity, motor thrust...), so we tell the filter how much we trust this assumption with a parameter $q$ (process noise). And we tell it how noisy the sensor is with $r$ (measurement noise).

Each step the filter does two things:
1. **Predict** the next state using the model.
2. **Correct** the prediction using the new measurement, weighted by how much it trusts each.

Tuning $q$ is the main knob: small $q$ → strongly trust the model (smooth, may lag); large $q$ → strongly trust the measurements (responsive, noisier).

In [ ]:
def kalman(x, dt, q, r):
    F = np.array([[1, dt], [0, 1]])           # constant-velocity transition
    H = np.array([[1, 0]])                    # we only measure altitude
    Q = np.array([[dt**4/4, dt**3/2],
                  [dt**3/2, dt**2]]) * q      # process noise covariance
    R = np.array([[r]])                       # measurement noise covariance
    I = np.eye(2)

    state = np.array([x[0], 0.0])
    P = np.eye(2)
    out = np.empty_like(x)
    for i in range(len(x)):
        # Predict
        state = F @ state
        P = F @ P @ F.T + Q
        # Correct using the new measurement
        y_innov = x[i] - (H @ state)[0]
        S = (H @ P @ H.T + R)[0, 0]
        K = (P @ H.T).flatten() / S
        state = state + K * y_innov
        P = (I - np.outer(K, H)) @ P
        out[i] = state[0]
    return out

## Try it out

Move the sliders and see what happens:

- **Noise σ** — how noisy the simulated altimeter is.
- **EMA α** — small = smooth & laggy, large = fast & noisy.
- **SMA window** — number of samples averaged.
- **Kalman q** — how much the filter trusts the constant-velocity model.

Things to look for:
- Which filter follows the sharp turn at the apex best?
- Which one *lags* most when the rocket is accelerating?
- What happens to each filter when you crank the noise way up?

In [ ]:
@interact(
    noise=FloatSlider(min=0.0, max=10.0, step=0.1, value=2.0,
                      description='Noise σ (m)', continuous_update=False),
    show_ema=Checkbox(value=True,  description='Show EMA'),
    alpha=FloatSlider(min=0.01, max=1.0, step=0.01, value=0.10,
                     description='EMA α', continuous_update=False),
    show_sma=Checkbox(value=True,  description='Show SMA'),
    window=IntSlider(min=1, max=100, step=1, value=20,
                     description='SMA window', continuous_update=False),
    show_kalman=Checkbox(value=True,  description='Show Kalman'),
    q=FloatLogSlider(value=0.5, base=10, min=-5, max=5, step=0.1,
                     description='Kalman q', continuous_update=False),
)
def plot_filters(noise, show_ema, alpha, show_sma, window, show_kalman, q):
    noisy = true_sig + noise * unit_noise
    r = max(noise, 0.01) ** 2

    plt.figure(figsize=(11, 5))
    plt.plot(t, noisy,    color='lightgray', lw=0.8, label='Noisy')
    plt.plot(t, true_sig, 'k--', lw=1.2, label='True')

    if show_sma:
        plt.plot(t, sma(noisy, window),       color='royalblue', lw=2, label=f'SMA window={window}')
    if show_ema:
        plt.plot(t, ema(noisy, alpha),        color='crimson',   lw=2, label=f'EMA α={alpha:.2f}')
    if show_kalman:
        plt.plot(t, kalman(noisy, dt, q, r),  color='seagreen',  lw=2, label=f'Kalman q={q:.3g}')

    plt.xlabel('Time (s)')
    plt.ylabel('Altitude (m)')
    plt.title(f'EMA vs Moving Average vs Kalman Filter  (σ = {noise:.1f} m)')
    plt.legend(loc='upper right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()